# Generate Report Inventory, OOS, MAT, Store Performance

Notebook ini otomatis mengubah 6 file mentah mingguan/bulanan jadi file
**"REPORT INVENTORY, OOS, MAT, STORE PERFORMANCE [MTD ...]"** yang sudah
diformat lengkap dengan formula VLOOKUP dan heatmap, seperti file yang
biasa kamu buat manual.

**Cara pakai:**
1. Kumpulkan jadi satu ZIP:
   - `Summary_MTD_<Bulan>_<Tahun> ....xlsx`
   - `[OOS] By Toko.xlsx`
   - `List DS_Covered Invent_....xlsx`
   - `inventory_darkstore_....xlsx`
   - `[MAT] Transaksi All.xlsx`
   - `[MAT] Transaksi Excl Beanspot.xlsx`
   - (opsional, tapi disarankan) file **Report** periode sebelumnya —
     dipakai sebagai template format & formula. Kalau tidak disertakan,
     notebook akan pakai template bawaan.
2. Jalankan semua cell dari atas ke bawah (Runtime > Run all).
3. Saat diminta, upload ZIP tadi.
4. File hasil otomatis ke-download di akhir.

**Catatan:**
- Excel akan otomatis menghitung ulang semua formula saat file pertama
  kali dibuka — tidak perlu langkah tambahan.
- Di bagian 5, notebook akan mencetak peringatan kalau ada toko yang
  terdaftar di "List DS" tapi datanya belum ada di salah satu file lain
  (mis. toko baru yang belum tercatat di Inventory hari itu). Itu wajar
  dan akan muncul sebagai `#N/A` di baris toko tersebut pada sheet
  Report — bukan error dari notebooknya.
- Kalau mau jalankan notebook ini dua kali di sesi Colab yang sama
  (upload ZIP baru tanpa restart), tidak masalah — folder ekstraksi
  dibersihkan otomatis tiap run.


In [ ]:
# (Jalan sekali) pastikan openpyxl versi terbaru tersedia
!pip install -q --upgrade openpyxl


## 1. Fungsi-fungsi pipeline (tidak perlu diubah)

In [ ]:
"""
Generate Report Inventory, OOS, MAT, Store Performance
=======================================================
Pipeline ini mengambil 6 file mentah + 1 file "Report" periode sebelumnya
(dipakai sebagai template formula/format), lalu menghasilkan file Report
baru untuk periode berjalan.

Sumber -> Sheet tujuan:
  Summary_MTD_*.xlsx            (sheet KD_STORE)  -> Store Performance
  [OOS] By Toko.xlsx            (sheet Sheet 1)   -> OOS
  List DS_Covered Invent_*.xlsx (sheet List DS)   -> List DS
  inventory_darkstore_*.xlsx    (sheet Store, Remark Item=ALL) -> Inventory
  [MAT] Transaksi All.xlsx      (baris 'Average') -> MAT All
  [MAT] Transaksi Excl Beanspot.xlsx (baris 'Average') -> MAT Exc Beanspot
Sheet 'Report' -> backbone Branch/Store Code/Store Name di-refresh &
  di-sort ulang berdasarkan RP INVENTORY (desc); formula VLOOKUP dibiarkan
  sama seperti template (referensi ke Table1/2/5/6/8/25).
"""
import re
import gc
import datetime as dt
from copy import copy
import openpyxl
from openpyxl.utils import get_column_letter

MONTH_ID = ["Januari","Februari","Maret","April","Mei","Juni","Juli",
            "Agustus","September","Oktober","November","Desember"]

# ---------- loaders ----------
# Semua loader pakai read_only=True + tutup workbook manual + gc.collect()
# supaya kalau ada file mentah yang ternyata jauh lebih besar dari
# biasanya (mis. ada sheet tersembunyi berukuran ratusan MB), openpyxl
# tidak menahan memori lebih lama dari yang dibutuhkan.

def load_kd_store(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['KD_STORE']
    rows = [list(r) for r in ws.iter_rows(values_only=True) if any(v is not None for v in r)]
    wb.close(); del wb; gc.collect()
    return rows

def load_oos(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['Sheet 1']
    rows = [list(r) for r in ws.iter_rows(values_only=True) if any(v is not None for v in r)]
    wb.close(); del wb; gc.collect()
    # buang baris ringkasan seperti 'Grand Total' di akhir
    rows = [r for r in rows if str(r[0]).strip().lower() != 'grand total']
    return rows

def load_list_ds(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['List DS']
    rows = [list(r) for r in ws.iter_rows(values_only=True) if r[0] is not None]
    wb.close(); del wb; gc.collect()
    return [r[:6] for r in rows]

def load_inventory(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['Store']
    all_rows = [list(r) for r in ws.iter_rows(values_only=True)]
    wb.close(); del wb; gc.collect()
    period_text = all_rows[1][0]  # 'PERIODE 01-09-2026 SD 07-09-2026'
    header_idx = next(i for i,r in enumerate(all_rows) if r[0] == 'Branch Code')
    header = all_rows[header_idx]
    data = all_rows[header_idx+1:]
    remark_idx = header.index('Remark Item')
    branch_name_idx = header.index('Branch Name')
    filtered = [r for r in data if r[0] is not None and r[remark_idx] == 'ALL']
    keep_idx = [i for i in range(len(header)) if i not in (0, branch_name_idx, remark_idx)]
    new_header = ['No'] + [header[i] for i in keep_idx]
    out = [new_header]
    for n, r in enumerate(filtered, start=1):
        out.append([n] + [r[i] for i in keep_idx])
    m = re.search(r'(\d{2})-(\d{2})-(\d{4})\s+SD\s+(\d{2})-(\d{2})-(\d{4})', period_text)
    start = dt.date(int(m.group(3)), int(m.group(2)), int(m.group(1)))
    end = dt.date(int(m.group(6)), int(m.group(5)), int(m.group(4)))
    return out, start, end

def load_mat_long(path, value_label):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['Sheet 1']
    rows = [list(r) for r in ws.iter_rows(values_only=True)]
    wb.close(); del wb; gc.collect()
    codes = rows[0][1:]
    names = rows[1][1:]
    avg_row = next(r[1:] for r in rows[2:] if r[0] == 'Average')
    out = [['No', 'Kode Toko', 'Nama Toko', value_label]]
    for i, (code, name, val) in enumerate(zip(codes, names, avg_row), start=1):
        out.append([i, code, name, val])
    return out

# ---------- writer helpers ----------

def _copy_row_style(ws, src_row, dst_row, ncols):
    for c in range(1, ncols + 1):
        s = ws.cell(row=src_row, column=c)
        d = ws.cell(row=dst_row, column=c)
        d.font = copy(s.font)
        d.fill = copy(s.fill)
        d.border = copy(s.border)
        d.alignment = copy(s.alignment)
        d.number_format = s.number_format

def write_table_sheet(wb, sheet_name, table_name, rows):
    """rows[0] = header, rows[1:] = data. Menimpa area tabel di sheet,
    membesar/mengecil kalau jumlah baris data berubah.

    PENTING: sengaja TIDAK pakai ws.insert_rows()/ws.delete_rows(). Kalau
    sheet punya dimensi terformat yang jauh lebih besar dari datanya
    (mis. max_row mendekati 1.048.576 karena sisa format Excel di seluruh
    kolom), kedua fungsi itu membuat openpyxl menggeser jutaan baris
    kosong dan bisa menghabiskan RAM/CPU dalam hitungan detik. Growth
    ditangani dengan menulis baris baru setelah baris terakhir (tidak
    perlu geser apa pun), shrink ditangani dengan mengosongkan nilai sel
    kelebihan (bukan menghapus barisnya)."""
    ws = wb[sheet_name]
    table = ws.tables[table_name]
    min_col, min_row, max_col, max_row = openpyxl.utils.cell.range_boundaries(table.ref)
    old_data_rows = max_row - min_row  # tidak termasuk header
    new_data_rows = len(rows) - 1
    ncols = len(rows[0])

    for r_off, row in enumerate(rows):
        for c_off, val in enumerate(row):
            ws.cell(row=min_row + r_off, column=min_col + c_off, value=val)

    new_last_row = min_row + new_data_rows

    if new_data_rows > old_data_rows:
        style_src_row = max_row if old_data_rows > 0 else min_row + 1
        for r in range(max_row + 1, new_last_row + 1):
            _copy_row_style(ws, style_src_row, r, ncols)
    elif new_data_rows < old_data_rows:
        for r in range(new_last_row + 1, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(row=r, column=c, value=None)

    new_ref = f"{get_column_letter(min_col)}{min_row}:{get_column_letter(max_col)}{new_last_row}"
    table.ref = new_ref
    return new_last_row - min_row  # jumlah baris data yang baru

# ---------- Report sheet ----------

REPORT_FORMULA_COLS = {
    5: "=VLOOKUP(C{r},Table8[],6,FALSE)",
    6: "=VLOOKUP(C{r},Table8[],7,FALSE)",
    7: "=VLOOKUP(C{r},Table8[],11,FALSE)",
    8: "=VLOOKUP(C{r},Table8[],12,FALSE)",
    9: "=VLOOKUP(C{r},Table8[],8,FALSE)",
    10: "=VLOOKUP(C{r},Table25[[Kode Toko]:[MAT All]],3,FALSE)",
    11: "=VLOOKUP(C{r},Table2[[#All],[Kode Toko]:[MAT Exc Beanspot]],3,FALSE)",
    12: "=VLOOKUP(C{r},Table6[],4,FALSE)",
    13: "=VLOOKUP(C{r},Table6[],5,FALSE)",
    14: "=VLOOKUP(C{r},Table5[[Store Code]:[MAT % Beanspot]],8,FALSE)",
    15: "=VLOOKUP(C{r},Table5[[Store Code]:[MAT % Beanspot]],10,FALSE)",
    16: "=VLOOKUP(C{r},Table5[[Store Code]:[MAT % Beanspot]],13,FALSE)",
    17: "=VLOOKUP(C{r},Table5[[Store Code]:[MAT % Beanspot]],15,FALSE)",
    18: "=VLOOKUP(C{r},Table1[],6,FALSE)",
}

def _shift_conditional_formatting(ws, old_first, old_last, new_first, new_last):
    from openpyxl.formatting.formatting import ConditionalFormattingList
    new_cf = ConditionalFormattingList()
    for cf in ws.conditional_formatting:
        old_sqref = str(cf.sqref)
        new_ranges = []
        for rng in old_sqref.split():
            m = re.match(r'^([A-Z]+)(\d+):([A-Z]+)(\d+)$', rng)
            if m and int(m.group(2)) == old_first and int(m.group(4)) == old_last:
                new_ranges.append(f"{m.group(1)}{new_first}:{m.group(3)}{new_last}")
            else:
                new_ranges.append(rng)
        new_sqref = " ".join(new_ranges)
        for rule in cf.rules:
            new_cf.add(new_sqref, rule)
    ws.conditional_formatting = new_cf

def rebuild_report_sheet(wb, kd_store_rows, list_ds_rows, inventory_rows,
                          period_start, period_end):
    ws = wb['Report']

    kd_header = kd_store_rows[0]
    idx_kode = kd_header.index('KD_STORE')
    idx_branch = kd_header.index('NAMA_BRANCH')
    idx_name = kd_header.index('NAMA_STORE')
    kd_map = {r[idx_kode]: (r[idx_branch], r[idx_name]) for r in kd_store_rows[1:]}

    inv_header = inventory_rows[0]
    idx_inv_code = inv_header.index('Store Code')
    idx_inv_value = inv_header.index('Inventory Value')
    inv_value_map = {r[idx_inv_code]: r[idx_inv_value] for r in inventory_rows[1:]}

    store_codes = [r[0] for r in list_ds_rows[1:]]
    store_codes.sort(key=lambda code: inv_value_map.get(code, 0), reverse=True)

    OLD_FIRST, OLD_LAST = 6, 72
    old_n = OLD_LAST - OLD_FIRST + 1
    new_n = len(store_codes)
    new_last = OLD_FIRST + new_n - 1

    # sengaja tidak pakai insert_rows/delete_rows (lihat catatan di
    # write_table_sheet) -- tulis baris baru setelah baris lama, atau
    # kosongkan baris kelebihan, tanpa pernah menggeser sheet.
    for i, code in enumerate(store_codes):
        r = OLD_FIRST + i
        branch, name = kd_map.get(code, (None, None))
        ws.cell(row=r, column=2, value=branch)
        ws.cell(row=r, column=3, value=code)
        ws.cell(row=r, column=4, value=name)
        for col, tmpl in REPORT_FORMULA_COLS.items():
            ws.cell(row=r, column=col, value=tmpl.format(r=r))
        if new_n > old_n and r > OLD_LAST:
            _copy_row_style(ws, OLD_LAST, r, 18)

    if new_n < old_n:
        for r in range(new_last + 1, OLD_LAST + 1):
            for c in range(2, 19):
                ws.cell(row=r, column=c, value=None)

    if new_n != old_n:
        _shift_conditional_formatting(ws, OLD_FIRST, OLD_LAST, OLD_FIRST, new_last)

    mid_end = f"{period_end.day} {MONTH_ID[period_end.month-1]}"
    ws['B3'] = f"Report Inventory MTD {period_start.day} - {period_end.day} {MONTH_ID[period_end.month-1]} {period_end.year}"
    ws['J4'] = f"MAT (%) [{period_start.day} - {period_end.day} {MONTH_ID[period_end.month-1][:3]}]"
    ws['L4'] = f"OOS (%) [{period_start.day} - {period_end.day} {MONTH_ID[period_end.month-1][:3]}]"
    ws['N4'] = f"INVENTORY RECAP [{period_start.day} - {period_end.day} {MONTH_ID[period_end.month-1]}]"

def check_data_gaps(list_ds_rows, kd_store_rows, inventory_rows, oos_rows,
                     mat_all_rows, mat_exc_rows):
    """Cek toko yang ada di List DS (master toko yang harus di-cover) tapi
    tidak ketemu di salah satu sumber lain. Cetak peringatan kalau ada,
    supaya #N/A di Report sheet nanti jelas asalnya (data yang memang
    belum ada saat itu, bukan bug pipeline)."""
    ld_codes = [r[0] for r in list_ds_rows[1:]]
    sources = {
        'Store Performance (Summary_MTD)': set(r[0] for r in kd_store_rows[1:]),
        'Inventory': set(r[1] for r in inventory_rows[1:]),
        'OOS': set(r[0] for r in oos_rows[1:]),
        'MAT All': set(r[1] for r in mat_all_rows[1:]),
        'MAT Exc Beanspot': set(r[1] for r in mat_exc_rows[1:]),
    }
    any_gap = False
    for src_name, codes in sources.items():
        missing = [c for c in ld_codes if c not in codes]
        if missing:
            any_gap = True
            print(f'  PERINGATAN: {len(missing)} toko di List DS tidak ada di {src_name}: {missing}')
    if not any_gap:
        print('  Semua toko di List DS ada datanya di 5 sumber lain. Aman.')
    return any_gap

# ---------- main ----------

def build_report(template_path, path_summary_mtd, path_oos, path_list_ds,
                  path_inventory, path_mat_all, path_mat_exc, output_path):
    kd_store_rows = load_kd_store(path_summary_mtd)
    oos_rows = load_oos(path_oos)
    list_ds_rows = load_list_ds(path_list_ds)
    inventory_rows, period_start, period_end = load_inventory(path_inventory)
    mat_all_rows = load_mat_long(path_mat_all, 'MAT All')
    mat_exc_rows = load_mat_long(path_mat_exc, 'MAT Exc Beanspot')

    print('Cek kelengkapan data toko:')
    check_data_gaps(list_ds_rows, kd_store_rows, inventory_rows, oos_rows,
                     mat_all_rows, mat_exc_rows)

    wb = openpyxl.load_workbook(template_path)

    write_table_sheet(wb, 'Store Performance', 'Table8', kd_store_rows)
    write_table_sheet(wb, 'OOS', 'Table6', oos_rows)
    write_table_sheet(wb, 'List DS', 'Table1', list_ds_rows)
    write_table_sheet(wb, 'Inventory', 'Table5', inventory_rows)
    write_table_sheet(wb, 'MAT All', 'Table25', mat_all_rows)
    write_table_sheet(wb, 'MAT Exc Beanspot', 'Table2', mat_exc_rows)

    rebuild_report_sheet(wb, kd_store_rows, list_ds_rows, inventory_rows,
                          period_start, period_end)

    wb.save(output_path)
    wb.close(); del wb; gc.collect()
    return output_path, period_start, period_end

def _score(fname, must_all=(), must_any=(), must_not=()):
    low = fname.lower()
    if any(m.lower() in low for m in must_not):
        return None
    if not all(m.lower() in low for m in must_all):
        return None
    if must_any and not any(m.lower() in low for m in must_any):
        return None
    return True

def autodetect_files(file_paths):
    """file_paths: daftar path hasil ekstrak zip (termasuk zip bersarang).
    Mengembalikan dict dengan key: template, summary_mtd, oos, list_ds,
    inventory, mat_all, mat_exc (None kalau tidak ketemu)."""
    import os
    result = {k: None for k in
               ['template', 'summary_mtd', 'oos', 'list_ds', 'inventory',
                'mat_all', 'mat_exc']}
    for p in file_paths:
        if not p.lower().endswith('.xlsx'):
            continue
        fn = os.path.basename(p)
        if _score(fn, must_all=['report'], must_any=['inventory', 'oos', 'mat', 'store performance']):
            result['template'] = p
        elif _score(fn, must_all=['summary_mtd']):
            result['summary_mtd'] = p
        elif _score(fn, must_all=['oos']):
            result['oos'] = p
        elif _score(fn, must_all=['list ds']):
            result['list_ds'] = p
        elif _score(fn, must_all=['inventory_darkstore']):
            result['inventory'] = p
        elif _score(fn, must_all=['mat'], must_any=['excl']):
            result['mat_exc'] = p
        elif _score(fn, must_all=['mat'], must_not=['excl']):
            result['mat_all'] = p
    return result

def extract_all(zip_path, extract_to):
    """Ekstrak zip_path, termasuk zip yang ada di dalamnya (nested), lalu
    kembalikan daftar semua file hasil ekstraksi. Folder tujuan dibersihkan
    dulu supaya tidak tercampur sisa ekstraksi run sebelumnya (mis. kalau
    notebook dijalankan dua kali di sesi yang sama tanpa restart)."""
    import zipfile, os, shutil
    if os.path.isdir(extract_to):
        shutil.rmtree(extract_to)
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_to)
    all_files = []
    for root, _, files in os.walk(extract_to):
        for f in files:
            full = os.path.join(root, f)
            if f.lower().endswith('.zip'):
                sub_dir = full + '_extracted'
                all_files.extend(extract_all(full, sub_dir))
            else:
                all_files.append(full)
    return all_files

if __name__ == '__main__':
    import sys
    if len(sys.argv) == 9:
        build_report(*sys.argv[1:9])


## 2. Template bawaan (fallback)
Dipakai hanya kalau ZIP yang diupload tidak berisi file Report periode sebelumnya.

In [ ]:
import base64

_DEFAULT_TEMPLATE_B64 = """..."""

with open('default_template.xlsx', 'wb') as f:
    f.write(base64.b64decode(_DEFAULT_TEMPLATE_B64))
print('Template bawaan siap (default_template.xlsx)')


Template bawaan siap (default_template.xlsx)


## 3. Upload ZIP berisi file mentah periode berjalan

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]
print('File terupload:', zip_path)


Saving Report MTD 31 Aug 2026 (Trial).zip to Report MTD 31 Aug 2026 (Trial).zip
File terupload: Report MTD 31 Aug 2026 (Trial).zip


## 4. Ekstrak & deteksi otomatis setiap file

In [ ]:
import os

all_files = extract_all(zip_path, 'extracted_input')
detected = autodetect_files(all_files)

missing = [k for k, v in detected.items() if v is None and k != 'template']
if detected['template'] is None:
    print('File Report periode sebelumnya tidak ditemukan di ZIP -> pakai template bawaan.')
    detected['template'] = 'default_template.xlsx'

print('File yang terdeteksi (ukuran file mentah, cek kalau ada yang janggal besar):')
for k, v in detected.items():
    if v:
        size_mb = os.path.getsize(v) / 1e6
        print(f'  {k:12s}: {v.split(chr(47))[-1]}  ({size_mb:.1f} MB)')
    else:
        print(f'  {k:12s}: TIDAK DITEMUKAN')

if missing:
    raise SystemExit(
        f'Ada file yang tidak ketemu di ZIP: {missing}. '
        'Cek lagi nama filenya, lalu upload ulang.'
    )


File Report periode sebelumnya tidak ditemukan di ZIP -> pakai template bawaan.
File yang terdeteksi (ukuran file mentah, cek kalau ada yang janggal besar):
  template    : default_template.xlsx  (0.1 MB)
  summary_mtd : Summary_MTD_August_2026 alfendio.a.faudisyah@gli.id.xlsx  (0.0 MB)
  oos         : [OOS] By Toko.xlsx  (0.0 MB)
  list_ds     : List DS_Covered Invent_26 Aug 26.xlsx  (0.0 MB)
  inventory   : inventory_darkstore_30-08-2026_sd_30-08-2026.xlsx  (53.3 MB)
  mat_all     : [MAT] Transaksi All.xlsx  (0.0 MB)
  mat_exc     : [MAT] Transaksi Excl Beanspot.xlsx  (0.0 MB)


## 5. Generate report

In [ ]:
tmp_output = '_report_tmp.xlsx'

out_path, period_start, period_end = build_report(
    detected['template'], detected['summary_mtd'], detected['oos'],
    detected['list_ds'], detected['inventory'], detected['mat_all'],
    detected['mat_exc'], tmp_output,
)

output_name = (
    f"REPORT INVENTORY, OOS, MAT, STORE PERFORMANCE "
    f"[MTD {period_end.day} {MONTH_ID[period_end.month-1][:3]} {period_end.year}].xlsx"
)
os.rename(out_path, output_name)
print('Selesai! File dibuat:', output_name)


Cek kelengkapan data toko:
  PERINGATAN: 1 toko di List DS tidak ada di Store Performance (Summary_MTD): ['BJ57']
  PERINGATAN: 2 toko di List DS tidak ada di Inventory: ['CI99', 'BJ57']
  PERINGATAN: 1 toko di List DS tidak ada di OOS: ['BJ57']
  PERINGATAN: 1 toko di List DS tidak ada di MAT All: ['BJ57']
  PERINGATAN: 1 toko di List DS tidak ada di MAT Exc Beanspot: ['BJ57']
Selesai! File dibuat: REPORT INVENTORY, OOS, MAT, STORE PERFORMANCE [MTD 30 Agu 2026].xlsx


## 6. Download hasilnya

In [ ]:
files.download(output_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>